## TF-IDF from scratch

Определим по тексту SMS - это spam или обычное сообщение.
текст → TF → IDF → TF-IDF матрица → Logistic Regression → prediction

In [1]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [2]:
documents = ["buy cheap iphone", 
             "buy cheap macbook", 
             "iphone for sale", 
             "meeting at office", 
             "office meeting tomorrow"
            ]
documents

['buy cheap iphone',
 'buy cheap macbook',
 'iphone for sale',
 'meeting at office',
 'office meeting tomorrow']

Заодно быстро вспомним про bag of words

TF-IDF работает со словами, поэтому сначала компьютер должен превратить текст в числа.

In [3]:
count_vectorizer = CountVectorizer()
X_count = count_vectorizer.fit_transform(documents) #числа
vocabulary = count_vectorizer.get_feature_names_out() # метод для получения словаря
print(vocabulary)

['at' 'buy' 'cheap' 'for' 'iphone' 'macbook' 'meeting' 'office' 'sale'
 'tomorrow']


In [4]:
bow_df = pd.DataFrame(
    X_count.toarray(),
    columns = vocabulary,
    index = documents
)

bow_df 
#Превращает весь текст в один вектор, 
# где каждый элемент вектора - это частота конкретного слова из словаря

,at,buy,cheap,for,iphone,macbook,meeting,office,sale,tomorrow
buy cheap iphone,0,1,1,0,1,0,0,0,0,0
buy cheap macbook,0,1,1,0,0,1,0,0,0,0
iphone for sale,0,0,0,1,1,0,0,0,1,0
meeting at office,1,0,0,0,0,0,1,1,0,0
office meeting tomorrow,0,0,0,0,0,0,1,1,0,1


TF — Term Frequency

Насколько часто слово встречается в документе

IDF — Inverse Document Frequency

Насколько слово редкое во всём корпусе

In [5]:
tokenized_documents = [
    document.lower().split()
    for document in documents
]

tokenized_documents

[['buy', 'cheap', 'iphone'],
 ['buy', 'cheap', 'macbook'],
 ['iphone', 'for', 'sale'],
 ['meeting', 'at', 'office'],
 ['office', 'meeting', 'tomorrow']]

In [6]:
vocabulary = sorted(
    set(
        word
        for document in tokenized_documents
        for word in document
    )
)

vocabulary

['at',
 'buy',
 'cheap',
 'for',
 'iphone',
 'macbook',
 'meeting',
 'office',
 'sale',
 'tomorrow']

In [7]:
word_to_idx = {
    word: idx
    for idx, word in enumerate(vocabulary)
}

word_to_idx

{'at': 0,
 'buy': 1,
 'cheap': 2,
 'for': 3,
 'iphone': 4,
 'macbook': 5,
 'meeting': 6,
 'office': 7,
 'sale': 8,
 'tomorrow': 9}

In [10]:
# tf = count(w, d)/ length(d), те это словарь частот слова в документе
# count(w,d) - кол-во слова w в документе d
# length(d) - длина документа

from collections import Counter

def compute_tf(document, vocabulary):
    word_counts = Counter(document)
    document_length = len(document)
    tf = {}
    for word in vocabulary:
        tf[word] = word_counts[word] / document_length
    return tf

compute_tf(tokenized_documents[0], vocabulary)

{'at': 0.0,
 'buy': 0.3333333333333333,
 'cheap': 0.3333333333333333,
 'for': 0.0,
 'iphone': 0.3333333333333333,
 'macbook': 0.0,
 'meeting': 0.0,
 'office': 0.0,
 'sale': 0.0,
 'tomorrow': 0.0}

In [13]:
# idf = size(c) / count(w, c), словарь
# size(c) - размер коллекции
# count(w, c) - количество документов в коллекции c, в которых встерчается слово w

# Теперь нужно понять, в скольких документах встречается каждое слово - count.
def compute_df(tokenized_documents, vocabulary):
    df = {}
    for word in vocabulary:
        document_count = 0
        for document in tokenized_documents:
            if word in document:
                document_count += 1

        df[word] = document_count
    return df

df = compute_df(
    tokenized_documents,
    vocabulary
)

df

{'at': 1,
 'buy': 2,
 'cheap': 2,
 'for': 1,
 'iphone': 2,
 'macbook': 1,
 'meeting': 2,
 'office': 2,
 'sale': 1,
 'tomorrow': 1}

Теперь считаем IDF

Используем простую формулу: 
IDF = ln(N / DF(w))

N - кол-во документов

In [15]:
def compute_idf(tokenized_documents, vocabulary):
    N = len(tokenized_documents)

    df = compute_df(tokenized_documents, vocabulary)

    idf = {}

    for word in vocabulary:
        idf[word] = np.log(N/df[word])

    return idf

idf = compute_idf(
    tokenized_documents,
    vocabulary
)

idf

{'at': np.float64(1.6094379124341003),
 'buy': np.float64(0.9162907318741551),
 'cheap': np.float64(0.9162907318741551),
 'for': np.float64(1.6094379124341003),
 'iphone': np.float64(0.9162907318741551),
 'macbook': np.float64(1.6094379124341003),
 'meeting': np.float64(0.9162907318741551),
 'office': np.float64(0.9162907318741551),
 'sale': np.float64(1.6094379124341003),
 'tomorrow': np.float64(1.6094379124341003)}

То есть:

macbook -более редкое слово - IDF больше

buy - более частое слово - IDF меньше

In [16]:
idf_df = pd.DataFrame({
    "word": vocabulary,
    "df": [df[word] for word in vocabulary],
    "idf": [idf[word] for word in vocabulary]
})

idf_df.sort_values(
    "idf",
    ascending=False
)

,word,df,idf
0,at,1,1.609438
3,for,1,1.609438
8,sale,1,1.609438
5,macbook,1,1.609438
9,tomorrow,1,1.609438
1,buy,2,0.916291
4,iphone,2,0.916291
2,cheap,2,0.916291
7,office,2,0.916291
6,meeting,2,0.916291


Наконец считаем TF-IDF

TF-IDF(w,d) = TF(w,d) ⋅ IDF(w)

w - слово
d - документ

In [17]:
def compute_tfidf(document, vocabulary, idf):
    tf = compute_tf(document,vocabulary)

    tfidf = {}

    for word in vocabulary:
        tfidf[word] = (tf[word] * idf[word])

    return tfidf

compute_tfidf(
    tokenized_documents[2],
    vocabulary,
    idf
)

{'at': np.float64(0.0),
 'buy': np.float64(0.0),
 'cheap': np.float64(0.0),
 'for': np.float64(0.5364793041447),
 'iphone': np.float64(0.3054302439580517),
 'macbook': np.float64(0.0),
 'meeting': np.float64(0.0),
 'office': np.float64(0.0),
 'sale': np.float64(0.5364793041447),
 'tomorrow': np.float64(0.0)}

In [18]:
tfidf_matrix = []

for document in tokenized_documents:

    tfidf = compute_tfidf(
        document,
        vocabulary,
        idf
    )

    row = [
        tfidf[word]
        for word in vocabulary
    ]

    tfidf_matrix.append(row)

tfidf_matrix = np.array(
    tfidf_matrix
)

tfidf_matrix.shape

(5, 10)

Получается 5 документов и 10 уникальных слов

In [19]:
tfidf_df = pd.DataFrame(
    tfidf_matrix,
    columns=vocabulary,
    index=documents
)

tfidf_df.round(3)

,at,buy,cheap,for,iphone,macbook,meeting,office,sale,tomorrow
buy cheap iphone,0.000,0.305,0.305,0.000,0.305,0.000,0.000,0.000,0.000,0.000
buy cheap macbook,0.000,0.305,0.305,0.000,0.000,0.536,0.000,0.000,0.000,0.000
iphone for sale,0.000,0.000,0.000,0.536,0.305,0.000,0.000,0.000,0.536,0.000
meeting at office,0.536,0.000,0.000,0.000,0.000,0.000,0.305,0.305,0.000,0.000
office meeting tomorrow,0.000,0.000,0.000,0.000,0.000,0.000,0.305,0.305,0.000,0.536
